# LLM JSON Classifier Dry Contract

Visible checkpoint for the first downloaded-model classifier ladder transition beyond BGE.

This notebook reads saved dev-only artifacts. It does not read holdout, train, load a local LLM, call a local LLM, or generate answers.

In [1]:
from pathlib import Path
import json

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

EMBED_DIR = PROJECT_ROOT / "data" / "eval" / "embedding_classifier_dev"
LLM_DRY_DIR = PROJECT_ROOT / "data" / "eval" / "llm_json_classifier_dry_run"

PROJECT_ROOT

WindowsPath('c:/Users/nwagb/Desktop/it_support_ai')

## Qwen3 Embedding Classifier Sample

These are bounded dev samples only. They confirm the Qwen3 embedding classifier path works on CUDA, but they are not full-dev model selection evidence.

In [2]:
summary_paths = [
    EMBED_DIR / "qwen3_embedding_06b" / "embedding_classifier_summary_dev_sample_text_qwen3_embedding_06b.json",
    EMBED_DIR / "qwen3_embedding_06b" / "embedding_classifier_summary_dev_sample_metadata_qwen3_embedding_06b.json",
    EMBED_DIR / "bge_small_en_v15" / "embedding_classifier_summary_dev_full_text_bge_small_en_v15.json",
    EMBED_DIR / "bge_small_en_v15" / "embedding_classifier_summary_dev_full_metadata_bge_small_en_v15.json",
]

rows = []
for path in summary_paths:
    summary = json.loads(path.read_text(encoding="utf-8"))
    rows.append(
        {
            "model": summary["model"]["key"],
            "scope": summary["scope"],
            "profile": summary["primary_metrics"]["profile"],
            "device": summary["model"]["resolved_device"],
            "eval_cases": summary["counts"]["eval_cases"],
            "reference_cases": summary["counts"]["reference_cases"],
            "primary_accuracy": summary["primary_metrics"]["accuracy"],
            "primary_macro_f1": summary["primary_metrics"]["macro_f1"],
            "multilabel_micro_f1": summary["multilabel_metrics"]["micro_f1"],
            "primary_hit_at_3": summary["ranked_metrics"]["primary_hit_at_3"],
            "runtime_seconds": summary["runtime_seconds"]["total"],
        }
    )

pd.DataFrame(rows)

,model,scope,profile,device,eval_cases,reference_cases,primary_accuracy,primary_macro_f1,multilabel_micro_f1,primary_hit_at_3,runtime_seconds
0,qwen3_embedding_06b,dev_sample,embedding_knn_text_only_qwen3_embedding_06b,cuda,32,128,0.8125,0.7842,0.7727,0.9375,34.803
1,qwen3_embedding_06b,dev_sample,embedding_knn_metadata_augmented_qwen3_embeddi...,cuda,32,128,0.8125,0.7842,0.7982,0.9375,52.609
2,bge_small_en_v15,dev_full,embedding_knn_text_only_bge_small_en_v15,cpu,1042,1042,0.8301,0.7246,0.7892,0.9760,79.788
3,bge_small_en_v15,dev_full,embedding_knn_metadata_augmented_bge_small_en_v15,cpu,1042,1042,0.8532,0.7556,0.7950,0.9818,84.189


## Dry LLM Contract Artifacts

The request rows are what a future local Gemma/Qwen classifier would consume. The expected labels are stored in a separate eval-key file for later dev-only scoring.

In [3]:
dry_summary_path = LLM_DRY_DIR / "llm_json_classifier_dry_run_summary_dev_sample_metadata.json"
dry_summary = json.loads(dry_summary_path.read_text(encoding="utf-8"))

pd.DataFrame([dry_summary["counts"]])

,available_routing_dev_cases,request_cases,eval_key_cases
0,1042,24,24


In [4]:
requests_path = PROJECT_ROOT / dry_summary["outputs"]["requests"]
eval_keys_path = PROJECT_ROOT / dry_summary["outputs"]["eval_keys"]

first_request = json.loads(requests_path.read_text(encoding="utf-8").splitlines()[0])
first_eval_key = json.loads(eval_keys_path.read_text(encoding="utf-8").splitlines()[0])
request_text = json.dumps(first_request, ensure_ascii=False)

assert '"expected_primary_domain"' not in request_text
assert '"expected_domains"' not in request_text
assert '"expected_behavior"' not in request_text
assert first_request["case_id"] == first_eval_key["case_id"]
assert "expected_primary_domain" in first_eval_key

{
    "request_case_id": first_request["case_id"],
    "request_has_expected_primary_key": '"expected_primary_domain"' in request_text,
    "eval_key_has_expected_primary_key": "expected_primary_domain" in first_eval_key,
    "schema_name": first_request["schema_name"],
}

{'request_case_id': 'stack_exchange_it_support_sites:serverfault:111746',
 'request_has_expected_primary_key': False,
 'eval_key_has_expected_primary_key': True,
 'schema_name': 'it_support_llm_classifier_v0'}

## Next Gate

Before running Gemma/Qwen inference, add the saved-response parser/evaluator. Then run a tiny explicitly approved local LLM smoke against the request artifact only.